#### From sql to delta
Notebook for tables extraction from * /Workspace/Users/carlitisamuel@gmail.com/chilean_news_analysys/DB/noticias.db * to delta tables

In [0]:
# Use chilean news catalog

spark.sql("USE CATALOG chilean_news")


In [0]:
# SOURCE DB EXPLORING
import sqlite3

conn = sqlite3.connect("/Workspace/Users/carlitisamuel@gmail.com/chilean_news_analysys/DB/noticias.db")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
for table in tables:
    print(table[0])

In [0]:
## CREATE BRONZE SCHEMA
# spark.sql("SELECT current_catalog()").show() #check current catalog
# spark.sql("CREATE SCHEMA IF NOT EXISTS chilean_news.bronze_tables")

In [0]:
noticas_raw_columns = [
    "author",
    "author_detail",
    "authors",
    "content",
    "df_fechap",
    "df_fechapl",
    "df_fid",
    "df_foto600",
    "df_horap",
    "df_tagnames",
    "df_ts",
    "guidislink",
    "id",
    "link",
    "links",
    "published",
    "published_parsed",
    "summary",
    "summary_detail",
    "tags",
    "title",
    "title_detail",
]


In [0]:


def read_sqlite_table(tablename:str):
    cursor.execute(f"SELECT * FROM {tablename}")
    tab = cursor.fetchall()
    cursor.execute(f"PRAGMA table_info({tablename})")
    cols = cursor.fetchall()
    columns = [col[1] for col in cols]
    return spark.createDataFrame(tab,schema=columns)


def create_table_from_df(df,tablename:str):
    df.write.mode("overwrite").saveAsTable(f"bronze_tables.{tablename}")

for table_name in tables:
    df = read_sqlite_table(table_name[0])
    df = df.dropDuplicates(["link"])
    create_table_from_df(df,table_name[0])
